# MisEdit_Audit.ipynb
**Paper:** MisEdit — Misconception Editing in Sub-3B Language Models

**Sections:**
- Section 1: MythBench dataset audit + 5-cluster assignment
- Section 2: EasyEdit setup + ROME compatibility (Qwen2.5-1.5B, TinyLlama-1.1B)
- Section 3: Evaluation functions (MC scoring v2)
- Section 4: Diagnostic — 5 misconceptions, pre/post generation + MC

---
## SECTION 1: MythBench Audit

In [ ]:
# Cell 1.1 — Load MythBench
import json, os

JSON_PATH = "/kaggle/input/datasets/kevinsam77/mythbench-dataset/MythBench_v10.json"

with open(JSON_PATH) as f:
    data = json.load(f)

misconceptions = data["misconceptions"]
controls = data["controls"]

print("=== Load successful ===")
print(f"Version: {data.get('version')}")
print(f"Note: {data.get('note')}")
print(f"Misconceptions: {len(misconceptions)}")
print(f"Controls: {len(controls)}")
print(f"Fields: {list(misconceptions[0].keys())}")

In [ ]:
# Cell 1.2 — 5-cluster assignment
CLUSTERS = {
    "Health_HumanBiology": [
        "antibiotics_kill_viruses", "sugar_causes_hyperactivity_children",
        "tongue_taste_zones", "hair_nails_grow_after_death",
        "carrots_improve_night_vision", "cold_weather_causes_colds",
        "cracking_knuckles_causes_arthritis", "humans_lose_most_heat_through_head",
        "humans_swallow_spiders_in_sleep", "coffee_dehydrates_you",
        "left_brain_right_brain_dominance", "shaving_makes_hair_grow_thicker",
        "alcohol_warms_you_up"
    ],
    "Animals_NaturalScience": [
        "bats_are_blind", "camels_store_water_in_humps",
        "bulls_enraged_by_red_color", "chameleons_camouflage_surroundings",
        "ostriches_bury_head_in_sand"
    ],
    "Physics_Earth_Space": [
        "great_wall_visible_from_space", "seasons_caused_by_distance_from_sun",
        "toilet_flush_coriolis_hemispheres", "glass_is_slow_liquid",
        "Mount_Everest_coldest_place_earth", "Mt_everest_highest_point_atmosphere",
        "water_drains_opposite_hemispheres", "diamond_hardest_material_earth"
    ],
    "History_Civilization": [
        "vikings_wore_horned_helmets", "marie_antoinette_let_them_eat_cake",
        "columbus_proved_earth_round", "cleopatra_was_egyptian",
        "nero_fiddled_while_rome_burned", "pilgrims_wore_black_and_white",
        "roman_vomitoriums_for_vomiting", "isaac_newton_apple_fell_on_head",
        "witch_trials_burned_at_stake_in_salem", "great_wall_built_to_keep_mongols_out",
        "ben_franklin_discovered_electricity", "lincoln_born_in_log_cabin_poverty",
        "cold_war_never_had_direct_combat", "signing_declaration_independence_july_4",
        "walt_disney_drew_mickey_mouse"
    ],
    "Geography_CulturalOrigins": [
        "fortune_cookies_chinese_origin", "french_fries_invented_in_france",
        "chinese_invented_pasta_marco_polo", "america_named_after_amerigo_vespucci",
        "alaska_is_northernmost_us_state", "pacific_ocean_named_for_being_calm",
        "mount_olympus_home_of_gods_in_clouds"
    ]
}

label_to_cluster = {}
for cluster, labels in CLUSTERS.items():
    for label in labels:
        label_to_cluster[label] = cluster

print("=== Cluster Assignment ===")
for cluster, labels in CLUSTERS.items():
    count = sum(1 for label in labels if any(m["misconception_label"] == label for m in misconceptions))
    print(f"  {cluster}: {count} items")

unassigned = [m["misconception_label"] for m in misconceptions if m["misconception_label"] not in label_to_cluster]
print(f"\nUnassigned: {len(unassigned)}")
total = sum(len(v) for v in CLUSTERS.values())
print(f"Total assigned: {total} / {len(misconceptions)}")
print(f"Controls available: {len(controls)}")

---
## SECTION 2: EasyEdit Setup + ROME Compatibility

In [ ]:
# Cell 2.1 — EasyEdit setup (Kaggle-safe, consolidated)
import sys

!git clone https://github.com/zjunlp/EasyEdit.git /kaggle/working/EasyEdit 2>/dev/null || echo "Already cloned"
sys.path.insert(0, '/kaggle/working/EasyEdit')

!pip install einops hydra-core higher omegaconf sentence-transformers peft \
             sentencepiece rouge av qwen-vl-utils iopath fairscale zhipuai -q

# Patch wikipedia dataset loader (deprecated in newer datasets library)
memit_file = "/kaggle/working/EasyEdit/easyeditor/models/rome/layer_stats.py"
with open(memit_file) as f:
    content = f.read()
content = content.replace(
    'dict(wikitext="wikitext-103-raw-v1", wikipedia="20200501.en")[ds_name]',
    'dict(wikitext="wikitext-103-raw-v1", wikipedia="wikitext-103-raw-v1")[ds_name]'
)
with open(memit_file, "w") as f:
    f.write(content)
print("layer_stats.py patched")

try:
    from easyeditor import BaseEditor
    print("EasyEdit: SUCCESS")
except ImportError as e:
    print(f"EasyEdit: FAILED — {e}")

import torch, transformers
print(f"Torch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
print(f"Transformers: {transformers.__version__}")

In [ ]:
# Cell 2.2 — Write hparams yamls for TinyLlama and Qwen2.5-1.5B
import yaml, shutil

hparams_path = "/kaggle/working/EasyEdit/hparams"

# TinyLlama — copy from llama3.2-3b, fix layer indices for 22-layer model
for method in ["ROME", "MEMIT"]:
    src = f"{hparams_path}/{method}/llama3.2-3b.yaml"
    dst = f"{hparams_path}/{method}/LlamaForCausalLM.yaml"
    shutil.copy(src, dst)
    with open(dst) as f:
        cfg = yaml.safe_load(f)
    cfg['model_name'] = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
    cfg['device'] = 0
    cfg['v_loss_layer'] = 21
    cfg['mom2_adjustment'] = False
    cfg['v_num_grad_steps'] = 10
    if 'layers' in cfg:
        cfg['layers'] = [i for i in cfg['layers'] if i < 22]
    with open(dst, "w") as f:
        yaml.dump(cfg, f)

# Qwen2.5-1.5B — copy from qwen2.5-7b, fix dims
for method in ["ROME", "MEMIT"]:
    src = f"{hparams_path}/{method}/qwen2.5-7b.yaml"
    dst = f"{hparams_path}/{method}/Qwen2ForCausalLM.yaml"
    shutil.copy(src, dst)
    with open(dst) as f:
        cfg = yaml.safe_load(f)
    cfg['model_name'] = 'Qwen/Qwen2.5-1.5B-Instruct'
    cfg['device'] = 0
    cfg['mom2_adjustment'] = False
    cfg['v_num_grad_steps'] = 10
    cfg['rewrite_module_tmp'] = 'model.layers.{}.mlp.down_proj'
    if method == "MEMIT":
        cfg['lm_head_module'] = 'model.embed_tokens'
    with open(dst, "w") as f:
        yaml.dump(cfg, f)

print("Hparams written for TinyLlama + Qwen2.5-1.5B (ROME + MEMIT)")

---
## SECTION 3: Evaluation Functions

In [ ]:
# Cell 3.1 — MC scoring v2 (option text probability, aligns with ROME edits)
import torch
import torch.nn.functional as F

def score_mc_belief_v2(model, tokenizer, question, options, correct_key, misconception_key):
    """
    Score each MC option by average log-probability of option text given question.
    Aligns with ROME's editing objective (text generation probability).
    """
    scores = {}
    question_prompt = f"{question}\nAnswer:"
    question_ids = tokenizer.encode(question_prompt, return_tensors="pt").to(model.device)

    for key, option_text in options.items():
        prompt = f"{question}\nAnswer: {option_text}"
        full_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
        option_len = full_ids.shape[1] - question_ids.shape[1]
        if option_len <= 0:
            scores[key] = -999.0
            continue
        with torch.no_grad():
            outputs = model(full_ids)
            logits = outputs.logits[0]
        log_probs = F.log_softmax(logits, dim=-1)
        option_start = question_ids.shape[1] - 1
        total = sum(
            log_probs[option_start + i, full_ids[0, question_ids.shape[1] + i].item()].item()
            for i in range(option_len)
        )
        scores[key] = total / option_len

    best_key = max(scores, key=scores.get)
    return {
        "correct_prob": scores.get(correct_key, -999.0),
        "misconception_prob": scores.get(misconception_key, -999.0),
        "predicted_key": best_key,
        "is_correct": best_key == correct_key,
        "is_misconception": best_key == misconception_key,
        "raw_scores": scores
    }


def validate_misconception_present(model, tokenizer, item, threshold=2):
    """
    3-prompt majority vote to confirm misconception is present pre-edit.
    Returns (present: bool, vote_count: int)
    """
    question = item["question"]
    options = item["options"]
    correct_key = item["correct"]
    misconception_key = item["misconception"]

    votes = 0
    # Prompt A: direct
    r = score_mc_belief_v2(model, tokenizer, question, options, correct_key, misconception_key)
    if r["is_misconception"]: votes += 1
    # Prompt B: True/False
    tf_q = f"True or False: {options[misconception_key]}"
    tf_opts = {"A": "True", "B": "False"}
    r2 = score_mc_belief_v2(model, tokenizer, tf_q, tf_opts, "B", "A")
    if r2["is_misconception"]: votes += 1
    # Prompt C: rephrase
    r3 = score_mc_belief_v2(model, tokenizer, f"Which is correct?\n{question}", options, correct_key, misconception_key)
    if r3["is_misconception"]: votes += 1

    return votes >= threshold, votes

print("Evaluation functions defined.")

---
## SECTION 4: Diagnostic — 5 Misconceptions, Pre/Post ROME

In [ ]:
# Cell 4.1 — Load 5 diagnostic misconceptions (one per cluster)
DIAG_LABELS = [
    "great_wall_visible_from_space",       # Physics_Earth_Space
    "bats_are_blind",                      # Animals_NaturalScience
    "antibiotics_kill_viruses",            # Health_HumanBiology
    "vikings_wore_horned_helmets",         # History_Civilization
    "seasons_caused_by_distance_from_sun"  # Physics_Earth_Space (control)
]

diag_items = [m for m in misconceptions if m["misconception_label"] in DIAG_LABELS]
print(f"Loaded {len(diag_items)} diagnostic items:")
for item in diag_items:
    print(f"  [{item['misconception_label']}] correct={item['correct']} misconception={item['misconception']}")

In [ ]:
# Cell 4.2 — Diagnostic function
from transformers import AutoTokenizer, AutoModelForCausalLM
from easyeditor import BaseEditor, ROMEHyperParams
import warnings
warnings.filterwarnings("ignore")

def run_diagnostic(model, tokenizer, editor, item):
    question = item["question"]
    options = item["options"]
    correct_key = item["correct"]
    misconception_key = item["misconception"]

    # Extract subject from question (rough heuristic)
    subject = item["misconception_label"].replace("_", " ")

    # PRE-EDIT: free generation
    inputs = tokenizer(question, return_tensors="pt").to(model.device)
    with torch.no_grad():
        gen_ids = model.generate(**inputs, max_new_tokens=30, do_sample=False,
                                  pad_token_id=tokenizer.eos_token_id)
    pre_gen = tokenizer.decode(gen_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    # PRE-EDIT: MC scoring
    pre_mc = score_mc_belief_v2(model, tokenizer, question, options, correct_key, misconception_key)

    # APPLY ROME
    _, edited_model, _ = editor.edit(
        prompts=[question],
        rephrase_prompts=[question],
        target_new=[options[correct_key]],
        subject=[subject],
        keep_original_weight=False
    )

    # POST-EDIT: free generation
    inputs2 = tokenizer(question, return_tensors="pt").to(edited_model.device)
    with torch.no_grad():
        gen_ids2 = edited_model.generate(**inputs2, max_new_tokens=30, do_sample=False,
                                          pad_token_id=tokenizer.eos_token_id)
    post_gen = tokenizer.decode(gen_ids2[0][inputs2["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    # POST-EDIT: MC scoring
    post_mc = score_mc_belief_v2(edited_model, tokenizer, question, options, correct_key, misconception_key)

    del edited_model
    torch.cuda.empty_cache()

    return {
        "label": item["misconception_label"],
        "pre_gen": pre_gen,
        "post_gen": post_gen,
        "pre_mc_predicted": pre_mc["predicted_key"],
        "pre_mc_correct_score": pre_mc["correct_prob"],
        "pre_mc_misc_score": pre_mc["misconception_prob"],
        "post_mc_predicted": post_mc["predicted_key"],
        "post_mc_correct_score": post_mc["correct_prob"],
        "post_mc_misc_score": post_mc["misconception_prob"],
        "gen_changed": pre_gen[:60] != post_gen[:60],
        "mc_predicted_changed": pre_mc["predicted_key"] != post_mc["predicted_key"],
        "pre_held_misconception": pre_mc["is_misconception"],
        "post_held_misconception": post_mc["is_misconception"],
    }

print("Diagnostic function defined.")

In [ ]:
# Cell 4.3 — Run diagnostic on Qwen2.5-1.5B + ROME
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
HPARAMS  = "hparams/ROME/Qwen2ForCausalLM.yaml"

tokenizer_q = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer_q.pad_token is None:
    tokenizer_q.pad_token = tokenizer_q.eos_token

model_q = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16, device_map="cuda:0", trust_remote_code=True
)

hparams_q = ROMEHyperParams.from_hparams(HPARAMS)
editor_q  = BaseEditor.from_hparams(hparams_q)

results_qwen = []
for item in diag_items:
    print(f"\n--- {item['misconception_label']} ---")
    r = run_diagnostic(model_q, tokenizer_q, editor_q, item)
    results_qwen.append(r)
    print(f"  PRE  gen:  {r['pre_gen'][:80]}")
    print(f"  POST gen:  {r['post_gen'][:80]}")
    print(f"  PRE  MC:   predicted={r['pre_mc_predicted']}  misc_score={r['pre_mc_misc_score']:.3f}")
    print(f"  POST MC:   predicted={r['post_mc_predicted']}  misc_score={r['post_mc_misc_score']:.3f}")
    print(f"  gen_changed={r['gen_changed']}  mc_changed={r['mc_predicted_changed']}")

del model_q
torch.cuda.empty_cache()

In [ ]:
# Cell 4.4 — Run diagnostic on TinyLlama-1.1B + ROME
MODEL_ID_T = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
HPARAMS_T  = "hparams/ROME/LlamaForCausalLM.yaml"

tokenizer_t = AutoTokenizer.from_pretrained(MODEL_ID_T, trust_remote_code=True)
if tokenizer_t.pad_token is None:
    tokenizer_t.pad_token = tokenizer_t.eos_token

model_t = AutoModelForCausalLM.from_pretrained(
    MODEL_ID_T, dtype=torch.float16, device_map="cuda:0", trust_remote_code=True
)

hparams_t = ROMEHyperParams.from_hparams(HPARAMS_T)
editor_t  = BaseEditor.from_hparams(hparams_t)

results_tinyllama = []
for item in diag_items:
    print(f"\n--- {item['misconception_label']} ---")
    r = run_diagnostic(model_t, tokenizer_t, editor_t, item)
    results_tinyllama.append(r)
    print(f"  PRE  gen:  {r['pre_gen'][:80]}")
    print(f"  POST gen:  {r['post_gen'][:80]}")
    print(f"  PRE  MC:   predicted={r['pre_mc_predicted']}  misc_score={r['pre_mc_misc_score']:.3f}")
    print(f"  POST MC:   predicted={r['post_mc_predicted']}  misc_score={r['post_mc_misc_score']:.3f}")
    print(f"  gen_changed={r['gen_changed']}  mc_changed={r['mc_predicted_changed']}")

del model_t
torch.cuda.empty_cache()

In [ ]:
# Cell 4.5 — Summary table
import pandas as pd

def make_summary(results, model_name):
    rows = []
    for r in results:
        rows.append({
            "Model": model_name,
            "Misconception": r["label"],
            "Pre held misc": r["pre_held_misconception"],
            "Post held misc": r["post_held_misconception"],
            "Gen changed": r["gen_changed"],
            "MC changed": r["mc_predicted_changed"],
            "Pre misc score": round(r["pre_mc_misc_score"], 3),
            "Post misc score": round(r["post_mc_misc_score"], 3),
        })
    return pd.DataFrame(rows)

df_q = make_summary(results_qwen, "Qwen2.5-1.5B")
df_t = make_summary(results_tinyllama, "TinyLlama-1.1B")
summary = pd.concat([df_q, df_t], ignore_index=True)

print("=== Diagnostic Summary ===")
print(summary.to_string(index=False))

print("\n=== Key Questions ===")
print(f"Cases where Gen changed but MC unchanged: {((summary['Gen changed']) & (~summary['MC changed'])).sum()}")
print(f"Cases where both changed: {(summary['Gen changed'] & summary['MC changed']).sum()}")
print(f"Cases where neither changed: {(~summary['Gen changed'] & ~summary['MC changed']).sum()}")

summary.to_csv("/kaggle/working/MisEdit_Diagnostic_Results.csv", index=False)
print("\nSaved: /kaggle/working/MisEdit_Diagnostic_Results.csv")